# 1. Construct downsampling dataset 

We downsampled SA5 with different non-OSN number from origin rds in 3 repeats. Cell composition background is retained. The result is saved as .rds.

In [ ]:
library(Seurat)

input_rds <- "/data/R02/tanj93/project/OSN_AmbientRNA_Benchmarking/20_8samples/8sample_rds/SA_5.rds"
seurat_obj <- readRDS(input_rds)

seurat_obj <- SetIdent(seurat_obj, value = "celltype")
counts <- GetAssayData(seurat_obj, assay="RNA", layer = "counts")
celltype <- seurat_obj$celltype

all_cells <- colnames(seurat_obj)
non_osn_cells <- colnames(subset(seurat_obj, idents = c("Mature OSNs","Immature OSNs"), invert = TRUE))
mature_osn_cells     <- colnames(subset(seurat_obj, idents = "Mature OSNs"))
immature_osn_cells <- colnames(subset(seurat_obj, idents = "Immature OSNs"))

cat("Total cells:", length(all_cells), "\n")
cat("non-OSN:", length(non_osn_cells), "\n")
cat("Mature OSN:", length(mature_osn_cells), "\n")
cat("Immature OSN:", length(immature_osn_cells), "\n")

# target non-OSN number
target_non_osn <- c(1200, 800, 400, 100)
n_repeats <- 3

out_root <- "/data/R04/zhangchao/joint_nuclei/03_analysis/SA5_downsampling"
set.seed(123) 

for (t in target_non_osn) {
  for (r in 1:n_repeats) {

    # 1. get a specified number of non OSN
    sampled_non_osn <- sample(non_osn_cells, t)

    # 2. Calculate the overall downsampling ratio
    down_ratio <- t / length(non_osn_cells)

    # 3. Extract Mature and Immature OSNs proportionally
    sampled_mature <- sample(mature_osn_cells, ceiling(length(mature_osn_cells) * down_ratio))
    sampled_immature <- sample(immature_osn_cells, ceiling(length(immature_osn_cells) * down_ratio))

    # 4. merge
    selected_cells <- c(sampled_non_osn, sampled_mature, sampled_immature)

    # 5. Create a Seurat object
    ds_counts <- counts[, selected_cells, drop = FALSE]
    ds_obj <- CreateSeuratObject(ds_counts)
    ds_obj$celltype <- celltype[selected_cells]

    # 6. save RDS
    out_dir <- file.path(out_root, paste0("SA5_", t), paste0("repeat_", r), "annotated_outs")
    dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)
    saveRDS(ds_obj, file.path(out_dir, "anno_ds.rds"))

    cat("Saved:", out_dir, 
        " non-OSN =", length(sampled_non_osn),
        " Mature OSN =", length(sampled_mature),
        " Immature OSN =", length(sampled_immature),
        " Total =", length(selected_cells), "\n")
  }
}


# 2.1 Generate data from annotated seurat object (.rds)

This function is used to transform the 'downsampled' seurat object to 3 kinds of file formats. 

Downsample cells based on annotated seurat object, in order to keep cells as subset of rawdata.

## Helper function


In [ ]:
library(Seurat)
library(DropletUtils)
library(sceasy)
library(Matrix)

export_downsampled_10x <- function(seurat_obj, outdir) {
  mat <- GetAssayData(seurat_obj, assay = "RNA", layer = "counts")
  barcodes <- colnames(mat)
  features <- rownames(mat)

  dir.create(outdir, showWarnings = FALSE, recursive = TRUE)

  # 1. trans annotated seurat object rds to cellrange out format folder
  mtx_dir <- file.path(outdir, "filtered_feature_bc_matrix")
  dir.create(mtx_dir, showWarnings = FALSE)

  writeMM(mat, file.path(mtx_dir, "matrix.mtx"))
  R.utils::gzip(file.path(mtx_dir, "matrix.mtx"), overwrite = TRUE)

  write.table(barcodes, file.path(mtx_dir, "barcodes.tsv"),
              quote = FALSE, row.names = FALSE, col.names = FALSE)
  R.utils::gzip(file.path(mtx_dir, "barcodes.tsv"), overwrite = TRUE)

  write.table(
    cbind(features, features, "Gene Expression"),
    file.path(mtx_dir, "features.tsv"),
    quote = FALSE, row.names = FALSE, col.names = FALSE, sep = "\t"
  )
  R.utils::gzip(file.path(mtx_dir, "features.tsv"), overwrite = TRUE)
  
  # 2. trans annotated seurat object rds to 10x h5 data, same as cellranger outs format
  DropletUtils::write10xCounts(
    file.path(outdir, "filtered_feature_bc_matrix.h5"),
    mat,
    gene.id = features,
    gene.symbol = features,
    overwrite = TRUE,
    version = "3"   # corresopnding to cellranger v3+
  )

  # 3. trans annotated seurat object rds to h5ad data
  sceasy::convertFormat(seurat_obj, from = "seurat", to = "anndata", outFile = file.path(outdir, "filtered_feature_bc_matrix.h5ad"))

  message("Export finished: ", outdir)
}

In [ ]:
library(Seurat)

root_dir <- "/data/R04/zhangchao/joint_nuclei/03_analysis/SA5_downsampling"
scales <- c(1200, 800, 400, 100)   
n_repeats <- 3

for (scale in scales) {
  for (rep in 1:n_repeats) {

    # Read the downsampled Seurat object
    rds_file <- file.path(root_dir, paste0("SA5_", scale), paste0("repeat_", rep), "annotated_outs", "anno_ds.rds")
    seurat_obj <- readRDS(rds_file)

    # output directory
    outdir <- file.path(root_dir, paste0("SA5_", scale), paste0("repeat_", rep), "annotated_outs")
    dir.create(outdir, recursive = TRUE, showWarnings = FALSE)

    export_downsampled_10x(seurat_obj = seurat_obj, outdir = outdir)

    cat("Exported data for nonOSN", scale, "repeat", rep, "\n")
  }
}

# 2.2 Generate data from cellranger outs

Filter cellranger outs 'filtered_feature_bc_matrix' accordding to barcode of downsampled data. Because CellClear uses raw and filtered of 'cellranger outs' as inputs.

In [ ]:
library(Seurat)
library(Matrix)
library(R.utils)

filtered_dir <- "/data/R02/tanj93/project/OSN_AmbientRNA_Benchmarking/01_rawdata/GSE157100/GSM4752989/cellranger/GSM4752989/outs/filtered_feature_bc_matrix"
filtered_counts <- Read10X(filtered_dir)

root_dir <- "/data/R04/zhangchao/joint_nuclei/03_analysis/SA5_downsampling"
scales <- c(1200, 800, 400, 100)   
n_repeats <- 3

for (scale in scales) {
  for (rep in 1:n_repeats) {

    # Read the downsampled Seurat object
    rds_file <- file.path(root_dir, paste0("SA5_", scale), paste0("repeat_", rep), "annotated_outs", "anno_ds.rds")
    seurat_obj <- readRDS(rds_file)

    # The name of the downsampled cells
    barcodes <- colnames(seurat_obj)
    
    # Filter filtered_counts to retain only the cells in the Seurat object and keep the order consistent
    filtered_subset <- filtered_counts[, barcodes]
    
    # gene names
    features <- rownames(filtered_subset)
    mat <- filtered_subset

    # output dir
    outdir <- file.path(root_dir, paste0("SA5_", scale), paste0("repeat_", rep), "cellranger_outs")
    mtx_dir <- file.path(outdir, "filtered_feature_bc_matrix")
    dir.create(mtx_dir, recursive = TRUE, showWarnings = FALSE)

    # write matrix.mtx
    writeMM(mat, file.path(mtx_dir, "matrix.mtx"))
    R.utils::gzip(file.path(mtx_dir, "matrix.mtx"), overwrite = TRUE)

    # write barcodes.tsv
    write.table(barcodes, file.path(mtx_dir, "barcodes.tsv"),
                quote = FALSE, row.names = FALSE, col.names = FALSE)
    R.utils::gzip(file.path(mtx_dir, "barcodes.tsv"), overwrite = TRUE)

    # write features.tsv
    write.table(
      cbind(features, features, "Gene Expression"),
      file.path(mtx_dir, "features.tsv"),
      quote = FALSE, row.names = FALSE, col.names = FALSE, sep = "\t"
    )
    R.utils::gzip(file.path(mtx_dir, "features.tsv"), overwrite = TRUE)

    cat("Saved Cell Ranger matrix for SA5_", scale, " repeat_", rep, "\n", sep="")
  }
}


# 3. Calculate contamination ratio and decontamination (corrected) matrix by different methods

## CellBender

We get subsets directly from the original results, because cellbender run on raw cellranger out 10x .h5 file, downsampling cell number does not influence the final results.

In [ ]:
# use a python script to run
from cellbender.remove_background.downstream import anndata_from_h5
import os

# 1. load all corrected data with full cell probability
cellbender_adata = anndata_from_h5("/data/R04/zhangchao/joint_nuclei/03_analysis/cellbender/SA5/cellbender_output_file.h5")  

scales = ["SA5_1200", "SA5_800", "SA5_400", "SA5_100"]
repeats = ["repeat_1", "repeat_2", "repeat_3"]

base_downsampling_dir = "/data/R04/zhangchao/joint_nuclei/03_analysis/SA5_downsampling"

for scale in scales:
    for rep in repeats:

        print(f"Processing {scale} {rep} ...")

        # 2. read downsampled barcodes
        ds_h5_path = os.path.join(base_downsampling_dir, scale, rep, "annotated_outs", "filtered_feature_bc_matrix.h5")
        if not os.path.exists(ds_h5_path):
            print(f"Warning: {ds_h5_path} not found, skipping.")
            continue

        ds_adata = anndata_from_h5(ds_h5_path)

        # 3. generate CellBender subset
        subset_adata = cellbender_adata[cellbender_adata.obs_names.isin(ds_adata.obs_names)].copy()

        # 4. save corrected matrix as h5ad
        out_h5ad = os.path.join(base_downsampling_dir, scale, rep, "cellbender", "cellbender_output_file_filtered.h5ad")
        os.makedirs(os.path.dirname(out_h5ad), exist_ok=True)
        subset_adata.write(out_h5ad)

        # 5. save contamination CSV
        filtered = subset_adata.obs[subset_adata.obs['cell_probability'] > 0.5]
        out_csv = os.path.join(base_downsampling_dir, scale, rep,  "cellbender_contamination_rate_percell.csv")
        filtered[['background_fraction']].to_csv(out_csv)

        print(f"Saved h5ad: {out_h5ad}")
        print(f"Saved contamination CSV: {out_csv}\n")


## FastCAR

Since FastCAR needs to find ambProfile based on raw_GEX, if we downsample cells for raw_GEX of original data, then raw_GEX equals to filtered_GEX. So we keep raw_GEX of original data as ambProfile.

In [ ]:
library(FastCAR)
library(Seurat)
library(sceasy)

raw_GEX <- Read10X("/data/R02/tanj93/project/OSN_AmbientRNA_Benchmarking/01_rawdata/GSE157100/GSM4752989/cellranger/GSM4752989/outs/raw_feature_bc_matrix/")

# ambient RNA profile
ambProfile <- describe.ambient.RNA.sequence(
  fullCellMatrix = raw_GEX,
  start = 10,
  stop = 500,
  by = 10,
  contaminationChanceCutoff = 0.05
)
emptyDropletCutoff <- recommend.empty.cutoff(ambProfile)
contaminationChanceCutoff <- 0.05
ambientProfile <- determine.background.to.remove(raw_GEX, emptyDropletCutoff, contaminationChanceCutoff)
ambient_genes <- names(ambientProfile[ambientProfile > 0])

scales <- c("SA5_1200", "SA5_800", "SA5_400", "SA5_100")
repeats <- c("repeat_1", "repeat_2", "repeat_3")
root <- "/data/R04/zhangchao/joint_nuclei/03_analysis/SA5_downsampling"

for (scale in scales) {
  for (rep in repeats) {
    cat("Processing:", scale, rep, "\n")
    
    seurat_rds <- file.path(root, scale, rep, "annotated_outs/anno_ds.rds")
    seurat_obj <- readRDS(seurat_rds)
    seurat_obj <- SetIdent(seurat_obj, value = "celltype")
    
    # OSN / non-OSN  
    osn_cells <- subset(seurat_obj, idents = "Mature OSNs")
    non_osn_cells <- subset(seurat_obj, idents = "Mature OSNs", invert = TRUE)
    
    osn_counts <- GetAssayData(osn_cells, slot = "counts")
    non_osn_counts <- GetAssayData(non_osn_cells, slot = "counts")
    
    # contamination rate
    if(length(ambient_genes) <= 1) {
      fastcar_non_osn_contamination_rate <- non_osn_counts[ambient_genes, ] / colSums(non_osn_counts)
      fastcar_osn_contamination_rate <- osn_counts[ambient_genes, ] / colSums(osn_counts)
    } else {
      fastcar_non_osn_contamination_rate <- colSums(non_osn_counts[ambient_genes, ]) / colSums(non_osn_counts)
      fastcar_osn_contamination_rate <- colSums(osn_counts[ambient_genes, ]) / colSums(osn_counts)
    }

    df_non_osn <- data.frame(
      barcodes = names(fastcar_non_osn_contamination_rate),
      fastcar = as.numeric(fastcar_non_osn_contamination_rate),
      celltype = "nonOSN",
      stringsAsFactors = FALSE
    )
    df_osn <- data.frame(
      barcodes = names(fastcar_osn_contamination_rate),
      fastcar = as.numeric(fastcar_osn_contamination_rate),
      celltype = "OSN",
      stringsAsFactors = FALSE
    )
    df_fastcar <- rbind(df_osn, df_non_osn)
    
    # save contamination rate CSV
    csv_out <- file.path(root, scale, rep, "fastcar_contamination_rate_percell.csv")
    write.csv(df_fastcar, csv_out, row.names = FALSE)
    
    # decontamination matrix
    filtered_GEX <- GetAssayData(seurat_obj, slot = "counts")
    fastcar_cellMatrix <- remove.background(filtered_GEX, ambientProfile)
    fastcar_seurat_obj <- CreateSeuratObject(fastcar_cellMatrix)
    
    # save matrix
    out_h5ad <- file.path(root, scale, rep, "fastcar/fastcar_corrected_mat.h5ad")
    dir.create(dirname(out_h5ad), recursive = TRUE, showWarnings = FALSE)
    sceasy::convertFormat(fastcar_seurat_obj, from = "seurat", to = "anndata", outFile = out_h5ad)
    
    cat("Finished:", scale, rep, "\n\n")
  }
}

# scAR

Based on downsampling .rds, we generate corresponding 10x .h5 file and rerun scar on it.

In [ ]:
#!/bin/bash

base="/data/R04/zhangchao/joint_nuclei/03_analysis/SA5_downsampling"
scales=("SA5_1200" "SA5_800" "SA5_400" "SA5_100")
repeats=("repeat_1" "repeat_2" "repeat_3")

for scale in "${scales[@]}"; do
  for rep in "${repeats[@]}"; do

    echo "====== Running SCAR: $scale / $rep ======"

    input="$base/$scale/$rep/annotated_outs/filtered_feature_bc_matrix.h5"
    output="$base/$scale/$rep/scar"

    mkdir -p "$output"

    scar "$input" \
      -ft mRNA \
      -o "$output" \
      -d cuda

    echo "------ Finished: $scale / $rep ------"
    echo ""

  done
done

In [ ]:
library(Seurat)
library(reticulate)

use_condaenv("/data/R04/zhangchao/anaconda3/envs/scar", required = TRUE)
py_config()
sc <- import("scanpy")

root_dir <- "/data/R04/zhangchao/joint_nuclei/03_analysis/SA5_downsampling"

scales  <- c("SA5_1200", "SA5_800", "SA5_400", "SA5_100")
repeats <- c("repeat_1", "repeat_2", "repeat_3")

for (scale in scales) {
  for (rep in repeats) {

    message("\n===== Processing SCAR contamination: ", scale, ", ", rep, " =====")

    # ---------- PATHS ----------
    rds_file  <- file.path(root_dir, scale, rep, "annotated_outs", "anno_ds.rds")
    h5ad_file <- file.path(root_dir, scale, rep, "scar", 
                           "filtered_feature_bc_matrix_denoised_mRNA.h5ad")
    out_file  <- file.path(root_dir, scale, rep, "scar_contamination_rate_percell.csv")

    # ---------- CHECK FILES ----------
    if (!file.exists(rds_file)) {
      warning("Missing Seurat file: ", rds_file)
      next
    }
    if (!file.exists(h5ad_file)) {
      warning("Missing SCAR h5ad: ", h5ad_file)
      next
    }

    # ---------- LOAD SEURAT ----------
    seurat_obj <- readRDS(rds_file)
    seurat_obj <- SetIdent(seurat_obj, value = "celltype")

    non_osn_cells <- subset(seurat_obj, idents = "Mature OSNs", invert = TRUE)
    osn_cells    <- subset(seurat_obj, idents = "Mature OSNs")

    # ---------- LOAD SCAR H5AD ----------
    adata <- sc$read_h5ad(h5ad_file)

    obs_names   <- rownames(adata$obs)
    noise_ratio <- adata$obs$noise_ratio
    names(noise_ratio) <- obs_names

    # all contamination
    all_contam <- noise_ratio

    # ---------- SPLIT ----------
    scar_non_osn <- all_contam[colnames(non_osn_cells)]
    scar_osn     <- all_contam[colnames(osn_cells)]

    # ---------- DF ----------
    df_scar <- rbind(
      data.frame(
        barcodes = names(scar_osn),
        scar = as.numeric(scar_osn),
        celltype = "OSN",
        stringsAsFactors = FALSE
      ),
      data.frame(
        barcodes = names(scar_non_osn),
        scar = as.numeric(scar_non_osn),
        celltype = "nonOSN",
        stringsAsFactors = FALSE
      )
    )

    # ---------- SAVE ----------
    write.csv(df_scar, out_file, row.names = FALSE)
    message("Saved: ", out_file)
  }
}


# scCDC

get corrected matrix from rds

In [ ]:
library(Seurat)

scales <- c(1200, 800, 400)
repeats <- 1:3
root_dir <- "/data/R02/tanj93/project/OSN_AmbientRNA_Benchmarking/20_8samples/num"

for (scale in scales) {
  for (rep in repeats) {
    path = file.path(root_dir,"corrected_rds", paste0("scCDC_SA5_",scale,"_rep",rep,".rds"))
    seurat_obj <- readRDS(path)

    counts_mat <- GetAssayData(seurat_obj, assay = "RNA", layer = "counts") # v5 style

    seurat_obj <- CreateSeuratObject(counts_mat)

    output_path <- file.path(
      "/data/R04/zhangchao/joint_nuclei/03_analysis/SA5_downsampling", 
      paste0("SA5_", scale), 
      paste0("repeat_", rep),
      paste0("scCDC/scCDC_",scale,"_corrected_mat.h5ad")
      )
    dir.create(dirname(output_path), recursive = TRUE, showWarnings = FALSE)

    sceasy::convertFormat(
        seurat_obj, 
        from = "seurat", 
        to = "anndata", 
        outFile = output_path, 
        main_layer = "counts"
    )
    message("Saved scCDC corrected h5ad to ", output_path)

  }
}


# CellClear

copy correted matrix folder

In [ ]:
import os
import shutil
import re

src_root = "/data/R02/tanj93/project/OSN_AmbientRNA_Benchmarking/20_8samples/num/corrected_rds"
dst_root = "/data/R04/zhangchao/joint_nuclei/03_analysis/SA5_downsampling"

# match as：75_cellclear_rep1_matrix
pattern = re.compile(r"cellclear_SA5_(\d+)_rep(\d+)_matrix")

for name in os.listdir(src_root):
    m = pattern.match(name)
    if not m:
        continue

    num = m.group(1)   # num
    rep = m.group(2)   # 1

    # source matrix folder
    src_matrix = os.path.join(src_root, name, "matrix")
    if not os.path.isdir(src_matrix):
        print(f"Skip: no matrix folder in {name}")
        continue

    # target dir
    dst_dir = os.path.join(
        dst_root,
        f"SA5_{num}",
        f"repeat_{rep}",
        "cellclear"
    )
    os.makedirs(dst_dir, exist_ok=True)

    # target matrix folder
    dst_matrix = os.path.join(dst_dir, "matrix")

    if os.path.exists(dst_matrix):
        shutil.rmtree(dst_matrix)

    shutil.copytree(src_matrix, dst_matrix)

    print(f"Copied {src_matrix} -> {dst_matrix}")

print("Done.")


# DecontX

get corrected matrix from rds

In [ ]:
library(Seurat)

scales <- c(1200, 800, 400)
repeats <- 1:3
root_dir <- "/data/R02/tanj93/project/OSN_AmbientRNA_Benchmarking/20_8samples/num"

for (scale in scales) {
  for (rep in repeats) {
    path = file.path(root_dir,"corrected_rds", paste0("decontx_SA5_",scale,"_rep",rep,".rds"))
    counts_mat <- readRDS(path)
    seurat_obj <- CreateSeuratObject(counts_mat)

    output_path <- file.path(
      "/data/R04/zhangchao/joint_nuclei/03_analysis/SA5_downsampling", 
      paste0("SA5_", scale), 
      paste0("repeat_", rep),
      paste0("decontx/decontx_",scale,"_corrected_mat.h5ad")
    )
    dir.create(dirname(output_path), recursive = TRUE, showWarnings = FALSE)

    sceasy::convertFormat(
        seurat_obj, 
        from = "seurat", 
        to = "anndata", 
        outFile = output_path, 
        main_layer = "counts"
    )
    message("Saved decontx corrected h5ad to ", output_path)

  }
}


# soupx 

get corrected matrix from rds

In [ ]:
library(Seurat)

scales <- c(1200, 800, 400)
repeats <- 1:3
root_dir <- "/data/R02/tanj93/project/OSN_AmbientRNA_Benchmarking/20_8samples/num"

for (scale in scales) {
  for (rep in repeats) {
    path = file.path(root_dir,"corrected_rds", paste0("SA5_",scale,"_rep",rep,"_soupx_corrected_counts.rds"))
    counts_mat <- readRDS(path)
    seurat_obj <- CreateSeuratObject(counts_mat)

    output_path <- file.path(
      "/data/R04/zhangchao/joint_nuclei/03_analysis/SA5_downsampling", 
      paste0("SA5_", scale), 
      paste0("repeat_", rep),
      paste0("soupx/soupx_",scale,"_corrected_mat.h5ad")
    )
    dir.create(dirname(output_path), recursive = TRUE, showWarnings = FALSE)

    sceasy::convertFormat(
        seurat_obj, 
        from = "seurat", 
        to = "anndata", 
        outFile = output_path, 
        main_layer = "counts"
    )
    message("Saved soupx corrected h5ad to ", output_path)

  }
}
